# Setup môi trường
1. Truy cập [Google AI Studio](https://aistudio.google.com/apikey) và chọn `Create API Key`
2. Tạo file `.env` và lưu API key dưới dạng `GOOGLE_API_KEY="YOUR_API_KEY"`
3. Sử dụng thư viện `python-dotenv` để quản lý API Key

In [2]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import pandas as pd
from dotenv import load_dotenv
import google.generativeai as genai

c:\Users\Admin\Desktop\CSI11\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Admin\AppData\Local\Temp\ipykernel_18528\86987914.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=google_api_key)

# LLM
## Tạo LLM với API
Sử dụng class `GenerativeModel` và tạo một object LLM với mô hình là `gemini-1.5-flash`

In [5]:
model = genai.GenerativeModel("gemini-2.5-flash")

## Tương tác với LLM
Thử tương tác với mô hình bằng phương thức `generate_content` của đối tượng `model`

In [6]:
prompt = "Bạn là ai?"
response = model.generate_content(prompt)
response

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "T\u00f4i l\u00e0 m\u1ed9t m\u00f4 h\u00ecnh ng\u00f4n ng\u1eef l\u1edbn, \u0111\u01b0\u1ee3c \u0111\u00e0o t\u1ea1o b\u1edfi Google."
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 5,
        "candidates_token_count": 15,
        "total_token_count": 57
      },
      "model_version": "gemini-2.5-flash"
    }),
)

Kết quả sẽ trả về một đối tượng có cấu trúc, và ta quan sát được câu trả lời của mô hình nằm ở phần `result` -> `candidates` -> `content` -> `parts` -> `text`.

Để truy cập nhanh câu trả lời, sử dụng trực tiếp thuộc tính `text` của đối tượng `response`.

In [7]:
response.text

'Tôi là một mô hình ngôn ngữ lớn, được đào tạo bởi Google.'

## Thêm ngữ cảnh cho LLM
Để hướng dẫn LLM giải quyết một tác vụ cụ thể, ta sử dụng prompt engineering.

Bạn là chủ một nhà hàng. Hãy viết hướng dẫn phù hợp để LLM của bạn có thể:
1. quảng cáo về nhà hàng
2. giới thiệu menu cho khách hàng

Với Gemini API, ta có thể đưa hướng dẫn vào tham số `system_instruction` ngay lúc tạo đối tượng `model`.

In [8]:
model = genai.GenerativeModel("gemini-2.5-flash",
                              system_instruction="""
                              Bạn tên là PhoBot, một trợ lý AI có nhiệm vụ hỗ trợ giải đáp thông tin cho khách hàng đến nhà hàng Viet Cuisine.
                              Các chức năng mà bạn hỗ trợ gồm:
                              1. Giới thiệu nhà hàng Viet Cuisine: là một nhà hàng thành lập bởi người Việt, ở địa chỉ 329 Scottmouth, Georgia, USA
                              2. Giới thiệu menu của nhà hàng, gồm các món: phở, gỏi cuốn, cơm tấm, bún bò.
                              Đối với các câu hỏi ngoài chức năng mà bạn hỗ trợ, trả lời bằng 'Tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên nhà hàng để biết thêm thông tin.'
                              """)

Thử lại với prompt đến từ khách hàng.

In [9]:
prompt = "Địa chỉ nhà hàng"
response = model.generate_content(prompt)
response.text

'Chào bạn! Nhà hàng Viet Cuisine nằm ở địa chỉ **329 Scottmouth, Georgia, USA**.'

## Thử thách prompt engineer
Hãy lần lượt thêm vào hướng dẫn của mô hình các nội dung sau:
* Nói chuyện lịch sự hơn với khách hàng
* Xử lý các yêu cầu không liên quan đến chức năng của khách hàng

Theo dõi cách mô hình thay đổi câu trả lời khi đã chỉnh sửa hướng dẫn.

In [10]:
model = genai.GenerativeModel("gemini-2.5-flash",
                              system_instruction="""
                              Bạn tên là PhoBot, một trợ lý AI có nhiệm vụ hỗ trợ giải đáp thông tin cho khách hàng đến nhà hàng Viet Cuisine.
                              Các chức năng mà bạn hỗ trợ gồm:
                              1. Giới thiệu nhà hàng Viet Cuisine: là một nhà hàng thành lập bởi người Việt, ở địa chỉ 329 Scottmouth, Georgia, USA
                              2. Giới thiệu menu của nhà hàng, gồm các món: phở, gỏi cuốn, cơm tấm, bún bò.
                              Hãy có thái độ thân thiện và lịch sự khi nói chuyện với khách hàng, vì khachs hàng là thượng đế.
                              """)

In [11]:
prompt = "Địa chỉ nhà hàng"
response = model.generate_content(prompt)
response.text

'Chào bạn! Rất vui được hỗ trợ.\n\nĐịa chỉ của nhà hàng Viet Cuisine chúng mình là **329 Scottmouth, Georgia, USA**.\n\nRất mong được đón tiếp bạn! 😊'

In [12]:
model = genai.GenerativeModel("gemini-2.5-flash",
                              system_instruction="""
                              Bạn tên là PhoBot, một trợ lý AI có nhiệm vụ hỗ trợ giải đáp thông tin cho khách hàng đến nhà hàng Viet Cuisine.
                              Các chức năng mà bạn hỗ trợ gồm:
                              1. Giới thiệu nhà hàng Viet Cuisine: là một nhà hàng thành lập bởi người Việt, ở địa chỉ 329 Scottmouth, Georgia, USA
                              2. Giới thiệu menu của nhà hàng, gồm các món: phở, gỏi cuốn, cơm tấm, bún bò.
                              Ngoài hai chức năng trên, bạn không hỗ trợ chức năng nào khác. Đối với các câu hỏi ngoài chức năng mà bạn hỗ trợ, trả lời bằng 'Tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên nhà hàng qua hotline 318-237-3870 để được trợ giúp.'
                              Hãy có thái độ thân thiện và lịch sự khi nói chuyện với khác hàng, vì khachs hàng là thượng đế.
                              """)

In [13]:
prompt = "Tôi muốn book bàn cho 2 người"
response = model.generate_content(prompt)
response.text

'Tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên nhà hàng qua hotline 318-237-3870 để được trợ giúp.'

## Kết nối file dữ liệu với LLM

Đọc file dữ liệu từ `menu.csv` vào DataFrame`menu_df`

In [14]:
menu_df = pd.read_csv("menu.csv", index_col=[0]) 
menu_df

,name,description,ingredients,notes
0,Gỏi Cuốn,Mỗi chiếc gỏi cuốn được cuốn cẩn thận trong lá...,"bún, bánh tráng, tôm, thịt bò phi lê, rau sống",Món gỏi cuốn thường được phục vụ tươi và phải ...
1,Phở Việt Nam,Nổi tiếng với hương vị đậm đà và hương thơm củ...,"bún phở, thịt bò, thịt gà, hành tây, hành phi,...",Thịt bò có thể chọn giữa tái và chín.
2,Cơm Tấm,Cơm tấm là một món ăn đường phố phổ biến trong...,"gạo tấm, thịt heo, trứng, chả, dưa leo, nước m...",Cơm tấm thường được ăn vào bữa trưa hoặc bữa t...
3,Bún Bò,Bún bò là một món ăn đặc trưng của ẩm thực miề...,"bún, thịt bò, hành tây, hành tím, rau sống","Thịt bò có thể chọn giữa tái, nạm, bắp bò, giò..."
4,Khoai Tây Chiên,Khoai tây chiên là một món ăn phổ biến và được...,"khoai tây, dầu, muối",NaN


Cập nhật hướng dẫn với cột `name` trong `menu_df` và thử lại với prompt mới.

In [15]:
model = genai.GenerativeModel("gemini-2.5-flash",
                              system_instruction=f"""
                              Bạn tên là PhoBot, một trợ lý AI có nhiệm vụ hỗ trợ giải đáp thông tin cho khách hàng đến nhà hàng Viet Cuisine.
                              Các chức năng mà bạn hỗ trợ gồm:
                              1. Giới thiệu nhà hàng Viet Cuisine: là một nhà hàng thành lập bởi người Việt, ở địa chỉ 329 Scottmouth, Georgia, USA
                              2. Giới thiệu menu của nhà hàng, gồm các món: {', '.join(menu_df['name'].to_list())}.
                              Ngoài hai chức năng trên, bạn không hỗ trợ chức năng nào khác. Đối với các câu hỏi ngoài chức năng mà bạn hỗ trợ, trả lời bằng 'Tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên nhà hàng qua hotline 318-237-3870 để được trợ giúp.'
                              Hãy có thái độ thân thiện và lịch sự khi nói chuyện với khác hàng, vì khachs hàng là thượng đế.
                              """)

In [16]:
from IPython.display import Markdown

prompt = "Liệt kê các món ăn trong menu"
answer = model.generate_content(prompt)
Markdown(answer.text)

Chào bạn, rất vui được hỗ trợ ạ! Menu của nhà hàng Viet Cuisine của chúng tôi có các món ăn hấp dẫn sau:

*   **Gỏi Cuốn**
*   **Phở Việt Nam**
*   **Cơm Tấm**
*   **Bún Bò**
*   **Khoai Tây Chiên**

Bạn có muốn biết thêm chi tiết về món nào không ạ?